In [40]:
# !kaggle datasets download -d lizhecheng/mistral-7b-instruct-v0-2-q6-k-gguf
# !unzip -o mistral-7b-instruct-v0-2-q6-k-gguf.zip -d /mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/pretrained_models
# !rm -rf mistral-7b-instruct-v0-2-q6-k-gguf.zip

# !kaggle datasets download -d gregremy/mistralv2
# !unzip -o mistralv2.zip -d /mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/pretrained_models
# !rm -rf mistralv2.zip


In [1]:
# ## Setup
import os

# os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [11]:
import os, sys, gc 
import pandas as pd
import numpy as np
import random
import time
from pathlib import Path 
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, GroupKFold, StratifiedGroupKFold 
from sklearn.preprocessing import OneHotEncoder
import argparse
import collections
import yaml
from types import SimpleNamespace
import re

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import SequentialSampler, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torch.utils.checkpoint



import transformers
from transformers import DebertaTokenizer, DebertaForTokenClassification
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import (
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
)

from functools import partial
import multiprocessing as mp
from scipy.special import softmax

from transformers import (AutoModelForTokenClassification,
                          AutoModelForCausalLM,
                          DataCollatorForTokenClassification, 
                          TrainingArguments, 
                          Trainer,
                          TrainerCallback)

import pytorch_lightning as pl
from pytorch_lightning import callbacks
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

import wandb
import warnings 
warnings.filterwarnings('ignore')

print('torch.__version__:', torch.__version__)
print('transformers.__version__:', transformers.__version__)
print('PLightning.__version__:', pl.__version__)


os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
# os.environ["NUMEXPR_NUM_THREADS"] = "1"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.__version__: 2.1.0+cu118
transformers.__version__: 4.33.3
PLightning.__version__: 2.0.9


In [46]:
class CFG:

    exp_name = 'exp04_fb2'
    task = "classification"

    n_folds = 10 #5 # 10
    trn_folds = [0] # [0,1,2,3,4]

    model_name = "mistralai/Mistral-7B-Instruct-v0.2"
    # "microsoft/deberta-base"
    # "allenai/longformer-base-4096"
    # "microsoft/deberta-large"

    llm_cache_dir = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/pretrained_models/"
    llm_name = "Mistral-7B-Instruct-v0.2"

    # max_length = 512 ##1536 #1024
    max_length = 1536 #768  
    stride = 128
    min_tokens = 6
    # TRAINING HYPERPARAMS
    batch_size = 16 #32 #4       # 16
    epochs = 4 #1
    pretrained_weights = ""
    
    GRAD_ACC = 1 #2 #4  #   8 # 4 ## 8
    LR = 2e-5    # 4e-5    ## 3e-05
    WD = 0.01
    WARMUP = 0.1
    use_custom_trainer = False # set True for custom training loop - TODO
    fp16 = True 
    seed = 42
    train = True
    debug = False # set True for debugging
    wandb = False
    wb_project = "NLP-Rater"
    output = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/"

    # ######
    # DATA_BASE_DIR = cfg["DATA_BASE_DIR"]
    # LR = cfg["LR"]
    # NUM_EPOCHS = cfg["NUM_EPOCHS"]
    NUM_CORES = 8 #cfg["NUM_CORES"]
    # BATCH_SIZE = cfg["BATCH_SIZE"]
    # USE_FP16 = cfg["USE_FP16"]
    # GRAD_ACCUM_STEPS = cfg["GRAD_ACCUM_STEPS"]
    # MAX_SEQ_LENGTH = cfg["MAX_SEQ_LENGTH"]
    # SPLIT_NUM = cargs.fold
    # PRETRAINED_MODEL = cargs.pretrained_model

    # TRAIN_CSV = os.path.join(DATA_BASE_DIR, 'feedback-prize-2021/train.csv')
    # TRAIN_DIR = os.path.join(DATA_BASE_DIR, 'feedback-prize-2021/train/')

In [ ]:
# with open(f'/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/config/exp04.yaml', 'r') as f:
#     config = yaml.safe_load(f)

# for k, v in config.items():
#     if type(v) == dict:
#         config[k] = SimpleNamespace(**v)
# cfg = SimpleNamespace(**config)
# print(cfg)

In [5]:
os.chdir('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3')
sys.path.append('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/')
sys.path.append('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/src/')

from src.utils.utils import *

DIR = Path("/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/input/")
OUTPUT = Path(CFG.output + CFG.exp_name)
OUTPUT.mkdir(exist_ok=True, parents=True)


# seed everything
CFG.device = device
set_seed(CFG.seed)

In [ ]:
if CFG.wandb:
    api_key = os.getenv('wb_api_key_core')
    wandb.login(key=api_key)
    wandb.init(project=CFG.wb_project, entity="core") #
    # anony = None

In [7]:
class_to_label_types = {
    'Claim': 0, 
    'Evidence': 1, 
    'Lead':2, 
    'Position':3, 
    'Concluding Statement':4,
    'Counterclaim':5, 
    'Rebuttal':6
}

label_types_to_class = {v:k for k, v in class_to_label_types.items()}

class_to_int = {
    'Claim': 0, 
    'Evidence': 1, 
    'Lead':2, 
    'Position':3, 
    'Concluding Statement':4,
    'Counterclaim':5, 
    'Rebuttal':6
}
int_to_class = {v:k for k, v in class_to_int.items()}

# Threshold found from CV
label_to_threshold = {
    0 : 0.275, #Claim
    1 : 0.375, #Evidence
    2 : 0.325, #Lead
    3 : 0.325, #Position
    4 : 0.4, #Concluding Statement
    5 : 0.275, #Counterclaim
    6 : 0.275 #Rebuttal
}



In [8]:
df_folds = pd.read_csv(DIR / 'train_fold10_sgkf.csv')

print('train folds shape:', df_folds.shape)
df_folds.head()

train folds shape: (144289, 18)


,essay_id,essay_text,discourse_id,discourse_start,discourse_end,discourse_type,predictionstring,discourse_text,discourse_effectiveness,discourse_type_num,hierarchical_id,hierarchical_text,hierarchical_label,holistic_essay_score,source_text,discourse_type_id,discourse_effectiveness_num,fold
0,423A1CA112E2,Phones\n\nModern humans today are always on th...,1622627660524,8,229,Lead,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,Modern humans today are always on their phone....,Non-Effective,Lead 1,NaN,NaN,NaN,3,NaN,5,0.0,2
1,423A1CA112E2,Phones\n\nModern humans today are always on th...,1622627653021,230,312,Position,45 46 47 48 49 50 51 52 53 54 55 56 57 58 59,They are some really bad consequences when stu...,Non-Effective,Position 1,NaN,NaN,NaN,3,NaN,6,0.0,2
2,423A1CA112E2,Phones\n\nModern humans today are always on th...,1622627671020,313,400,Evidence,60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75,Some certain areas in the United States ban ph...,Non-Effective,Evidence 1,1.622628e+12,They are some really bad consequences when stu...,Position,3,NaN,4,0.0,2
3,423A1CA112E2,Phones\n\nModern humans today are always on th...,1622627696365,401,756,Evidence,76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 9...,"When people have phones, they know about certa...",Non-Effective,Evidence 2,1.622628e+12,They are some really bad consequences when stu...,Position,3,NaN,4,0.0,2
4,423A1CA112E2,Phones\n\nModern humans today are always on th...,1622627759780,757,884,Claim,139 140 141 142 143 144 145 146 147 148 149 15...,Driving is one of the way how to get around. P...,Non-Effective,Claim 1,1.622628e+12,They are some really bad consequences when stu...,Position,3,NaN,1,0.0,2


# Dataset 

In [14]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, cfg, df, tokenizer=None, mode='train'):
        # self.input_dict = input_dict
        self.cfg = cfg  
        self.df = df
        self.mode = mode
        self.tokenizer = tokenizer

        self.texts = df['discourse_text'].values.tolist()

        if mode == 'train':
            self.labels = df['discourse_effectiveness_num'].values # effectiveness
            self.labels_type = df['discourse_type'].values 
        else:
            self.labels = np.zeros((len(df), 1))
            # self.labels_type = None
    
    def __len__(self):
        return len(self.df) #len(self.input_dict['input_ids'])
    
        
    def __getitem__(self, index):
        
        text = self.texts[index]
        targets = self.labels[index]
        
        ## tokenize the text
        o = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True)

        out = {}
        out['input_ids'] = o.input_ids
        out['attention_mask'] = o.attention_mask
        out['labels'] = torch.tensor(targets, dtype=np.float32)
        #out = {k:self.input_dict[k][index] for k in self.input_dict.keys() if k not in {'essay_id', 'offset_mapping', 'word_ids'}}
        return out
    
    def get_filename(self, index):
        return self.df.iloc[index]['essay_id_comp'] #self.input_dict['essay_id'][index]
    
    # def get_offset(self, index):
    #     return self.input_dict['offset_mapping'][index]


In [ ]:
### TODO 

def get_embedings_from_llm(texts, model, tokenizer):
    # model = AutoModelForTokenClassification.from_pretrained(CFG.model_name)
    # tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
    model = model.to(device)
    model.eval()
    embeds = []
    for i in range(0, len(texts), 1):
        input_ids = tokenizer(texts[i], return_tensors="pt", padding="max_length", truncation=True, max_length=CFG.max_length)['input_ids']
        input_ids = input_ids.to(device)
        with torch.no_grad():
            output = model(input_ids)
        embeds.append(output.last_hidden_state.cpu().numpy())
    return embeds

In [12]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

tokenizer.save_pretrained(OUTPUT / 'tokenizer')

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp04_fb2/tokenizer/tokenizer_config.json',
 '/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp04_fb2/tokenizer/special_tokens_map.json',
 '/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp04_fb2/tokenizer/tokenizer.json')

In [55]:
# # model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# model_name = os.path.join(CFG.llm_cache_dir, CFG.llm_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)

'/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/pretrained_models/Mistral-7B-Instruct-v0.2'

In [48]:
# Enhanced Classifier model - from Christina 

class RaterClf_01(nn.Module):
    def __init__(self, cfg, num_classes=2, dropout_rate=0.1, cache=True, vb=False):
        super(RaterClf_01, self).__init__()

        if cache:
            model_name = os.path.join(cfg.llm_cache_dir, cfg.llm_name)
        else:
            model_name = cfg.model_name
            # model_name = "mistralai/Mistral-7B-Instruct-v0.2"
    
        llm_model = AutoModelForCausalLM.from_pretrained(model_name)

        self.embedding_size  = llm_model.config.hidden_size
        self.embedding_layer = llm_model.get_input_embeddings()

        ### add sequential block
        # self.block1 = nn.Sequential()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(self.embedding_size, self.embedding_size)  # First fully connected layer
        self.bn1 = nn.BatchNorm1d(self.embedding_size)  # Batch normalization
        ### head 
        self.fc2 = nn.Linear(self.embedding_size, num_classes)  # Second fully connected layer

        # #### 
        # if self.cfg.loss_function == "CrossEntropy":
        #     self.loss_fn = DenseCrossEntropy()
        # elif self.cfg.loss_function == "WeightedCrossEntropy":
        #     self.loss_fn = WeightedDenseCrossEntropy()
        # elif self.cfg.loss_function == "FocalLoss":
        #     self.loss_fn = FocalLoss()
        # if self.cfg.loss_function == "BCE":
        #     self.loss_fn = nn.BCEWithLogitsLoss()
        # else:
        #     self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, batch):
        
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        bs = input_ids.size(0)
        if self.vb: print('input_ids:', input_ids.shape, 'attention_mask:', attention_mask.shape)

        embeddings = self.embedding_layer(input_ids)
        if self.vb: print('embeddings:', embeddings.shape)

        masked_embeddings = embeddings * attention_mask.unsqueeze(-1).float()
        pooled_embeddings = torch.mean(masked_embeddings, dim=1)
        ### TODO try other pooling methods

        x = self.dropout(pooled_embeddings)
        if bs > 1:  # Apply batch normalization only if batch size > 1
            x = self.bn1(x)
        x = F.relu(self.fc1(x))
        if self.vb: print('fc1:', x.shape)
        logits = self.fc2(x)
        return logits

In [49]:
class TextDataModule(pl.LightningDataModule):
    def __init__(self, cfg, 
        train_df = None,
        valid_df = None,
        predict_df = None,
        tokenizer = None,
    ):
        super().__init__()
        self.cfg = cfg
        self.train_df = train_df
        self.valid_df = valid_df
        self.predict_df = predict_df
        self.tokenizer = tokenizer
        
    def setup(self, stage):
        self.train_dataset = TextDataset(self.cfg, self.train_df, self.tokenizer, mode='train')
        self.valid_dataset = TextDataset(self.cfg, self.valid_df, self.tokenizer, mode='train')

    def train_dataloader(self):
        # custom_collator = CustomCollator(self.tokenizer)
        return DataLoader(
            self.train_dataset, 
            # **self.cfg["train_loader"], 
            # collate_fn=custom_collator,
            batch_size = self.cfg.batch_size,
            drop_last = True,
            num_workers=2, # 16
            pin_memory=False,
            shuffle=True,
            )

    def val_dataloader(self):
        # custom_collator = CustomCollator(self.tokenizer)
        return DataLoader(
            self.valid_dataset, 
            # collate_fn=custom_collator,
            batch_size = self.cfg.batch_size,
            drop_last = False,
            num_workers=2, # 16
            pin_memory=False,
            shuffle=False,
              )

    def predict_dataloader(self):
        # custom_collator = CustomCollator(self.tokenizer)
        return DataLoader(
            self.valid_dataset,  
            # collate_fn=custom_collator,
            # **self.cfg["val_loader"], 
            batch_size = 16,
            drop_last = False,
            num_workers=2, # 16
            pin_memory=False,
            shuffle=False,
            )
    

# Light Model 

In [31]:
def get_scheduler(cfg, optimizer, total_steps):
    if cfg.training.schedule == "Linear":
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(
                cfg.training.warmup_epochs * (total_steps // cfg.training.batch_size)
            ),
            num_training_steps=cfg.training.epochs
            * (total_steps // cfg.training.batch_size),
        )
    else:
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(
                cfg.training.warmup_epochs * (total_steps // cfg.training.batch_size)
            ),
            num_training_steps=cfg.training.epochs
            * (total_steps // cfg.training.batch_size),
        )
    return scheduler

In [50]:
# from torcheval.metrics import MulticlassF1Score  
from torchmetrics import Accuracy, Precision, Recall, AUROC, F1Score


class TextModel(pl.LightningModule):
    def __init__(self, cfg, config_path=None):
        super().__init__()

        self.cfg = cfg
        self.scorer = F1Score(task="multiclass", num_classes=2) # "binary", "multiclass", "multilabel"
        # self.model_cfg = cfg['model']

        # self.criterion = DenseCrossEntropy()
        # self.criterion = WeightedDenseCrossEntropy()
        # self.criterion = FocalLoss()
        self.criterion = nn.CrossEntropyLoss()
        self.model = RaterClf_01(cfg)
        self.skip_validation = 0

    def forward(self, inputs):
        # x = self.backbone(**inputs).last_hidden_state
        # x = self.dropout(x)
        out = self.model(inputs)
        return out

    def on_validation_epoch_start(self):
        print('\n')

    def on_validation_epoch_end(self):
        if self.skip_validation > 0:
            self.skip_validation -= 1

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch)

    def validation_step(self, batch, batch_idx):
        if self.skip_validation > 0:
            return {}
        return self.shared_step(batch)

    def shared_step(self, batch):
        # text, span_labels, label_ids, label_counts = batch
        targets = batch['labels']

        output = self(batch)
        # preds, new_span_labels, loss_divs = preds_to_span_preds(output, label_ids, span_labels, label_counts)
        
        if self.cfg.loss_fn=='CrossEntropy':
            preds = output.softmax(dim=1)
            loss = self.criterion(preds, targets)
        else:
            preds = output.sigmoid()
            loss = self.criterion(output, targets)

        preds = preds.detach().cpu()
        true_values = targets.detach().cpu()
        return {'loss': loss, 'preds': preds, 'trues': true_values}

    def predict_step(self, batch, batch_idx):
        # text, span_labels, label_ids = batch
        # output = self(text)
        output = self(batch)
        return output.detach().cpu()

    def validation_epoch_end(self, outputs):
        if self.skip_validation > 0:
            self.log(f'val_loss', 1.0, on_epoch=True, prog_bar=True)
            self.log(f'score', 1.0 * self.skip_validation, on_epoch=True, prog_bar=True)
            return

        preds = []
        trues = []
        loss = 0
        for out in outputs:
            if out['loss'] is not None:
                loss += out['loss']
                preds.extend([x for x in out['preds']])
                trues.extend([x for x in out['trues']])
        loss /= len(outputs)

        preds = torch.stack(preds)
        trues = torch.stack(trues)
        val_score = self.scorer(preds, trues)

        self.log(f'val_loss', loss, on_epoch=True, prog_bar=True)
        self.log(f'f1_score', val_score, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):

        weight_decay = self.cfg.training.weight_decay #self.cfg['optimizer']['weight_decay']
        param_optimizer = list(self.named_parameters())
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_parameters = [
            {
                'params': [p for n, p in param_optimizer if 'backbone' in n and not any(nd in n for nd in no_decay)],
                'weight_decay': weight_decay,
                'lr': self.cfg.training.learning_rate, # self.cfg['optimizer']['params']['lr'],
            },
            {
                'params': [p for n, p in param_optimizer if 'backbone' in n and any(nd in n for nd in no_decay)],
                'weight_decay': 0.0,
                'lr': self.cfg.training.learning_rate, #self.cfg['optimizer']['params']['lr'],
            },
            {
                'params': [p for n, p in param_optimizer if not 'backbone' in n and not any(nd in n for nd in no_decay)],
                'weight_decay': weight_decay,
                # 'lr': self.cfg['optimizer']['params']['lr'] * self.cfg['optimizer']['head_lr_factor'],
                'lr': self.cfg.training.learning_rate * 5.0,
            },
            {
                'params': [p for n, p in param_optimizer if not 'backbone' in n and any(nd in n for nd in no_decay)],
                'weight_decay': 0.0,
                # 'lr': self.cfg['optimizer']['params']['lr'] * self.cfg['optimizer']['head_lr_factor'],
                'lr': self.cfg.training.learning_rate * 5.0,
            },
        ]

        optimizer = torch.optim.AdamW(optimizer_parameters, lr=self.cfg.LR, eps = 1.0e-06, betas = [0.9, 0.999]) 
        scheduler = get_scheduler(self.cfg, optimizer, self.cfg.dataset_size)

        lr_scheduler_config = {
                'scheduler': scheduler,
                'interval': 'step', #self.cfg['scheduler']['interval'],
                'frequency': 1,
            }

        return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler_config}

In [51]:
# CFG.pretrained_weights = ""
CFG.batch_size = 4 #16

In [52]:
### def train_loop()

for fold in range(CFG.n_folds):

    # if not opt.fold is None and opt.fold != fold:
    if fold in CFG.trn_folds:
        continue

    train_df = df_folds[df_folds['fold'] != fold].reset_index(drop=True)
    valid_df = df_folds[df_folds['fold'] == fold].reset_index(drop=True)
    CFG.dataset_size = len(train_df)

    datamodule = TextDataModule(cfg=CFG, train_df=train_df, valid_df=valid_df, tokenizer=tokenizer, )

    lit_model = TextModel(CFG)
    torch.save(lit_model.config, OUTPUT / 'config.pth')

    # if CFG.pretrained_weights != "":
    #     pl_fold = random.randint(0, 7)
    #     print('USING PRETRAINED FOLD:', pl_fold)

    #     state_dict = torch.load(cfg.architecture.pretrained_weights) #torch.load(CFG.use_pretrained + '/fold' + str(pl_fold) + '.pth')
    #     del state_dict['fc.bias']
    #     del state_dict['fc.weight']
    #     del state_dict['fc_seg.bias']
    #     del state_dict['fc_seg.weight']

    #     for key in list(state_dict.keys()):
    #         state_dict[key.replace('model.deberta.', '')] = state_dict.pop(key)

    #     model.backbone.load_state_dict(state_dict, strict=True)

    earystopping = EarlyStopping(
        monitor = 'f1_score',
        patience = 2,  
        mode = 'max',
    )
    callback_list = [earystopping]

    ckp = callbacks.ModelCheckpoint(
        dirpath=OUTPUT / 'checkpoints',
        filename=f'{CFG.model_name.split("/")[-1]}_f{fold}' + '-{epoch}-{score:.4f}',
        monitor='f1_score',
        save_weights_only=True,
        save_top_k=1,
        mode='max', #'min',
        save_last=False,
    )
    callback_list.append(ckp)

    trainer = pl.Trainer(
        logger=None,
        max_epochs=CFG.epochs,
        callbacks=callback_list,
        accelerator='gpu',
        accumulate_grad_batches = 1,
        fast_dev_run = False,
        num_sanity_val_steps = 0,
        precision = 16 if CFG.fp16 else 32,
        val_check_interval = 0.1,
        # resume_from_checkpoint = null,
        # devices=opt.device,
        # **cfg['trainer'],
    )

    trainer.fit(lit_model, datamodule=datamodule)

KeyError: 'mistral'